In [8]:
# # GSS Option A - Debug Notebook
#
# ## Current Error Analysis
# **Error Location:** `step2_multistock_validation.py` line 154
#
# **Problem:** After merging `df` (5-min data) with `df_daily`, the column structure changes.
#
# ---


In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [10]:
# ## 1. Simulate the DataFrame Merge Issue
#
# Let's recreate what happens when you merge two DataFrames with overlapping column names.


In [11]:
# 5-min DataFrame (df)
df_5min = pd.DataFrame({
    'datetime': pd.date_range('2024-01-01 09:15', periods=5, freq='5min'),
    'Close': [100, 101, 102, 103, 104],
    'MA20': [99, 99.5, 100, 100.5, 101],
    'date': pd.date_range('2024-01-01', periods=5, freq='D').date
})

# Daily DataFrame (df_daily)
df_daily = pd.DataFrame({
    'datetime': pd.date_range('2024-01-01', periods=5, freq='D').date,
    'ma50': [98, 98.5, 99, 99.5, 100],
    'ma100': [97, 97.5, 98, 98.5, 99],
    'ma200': [96, 96.5, 97, 97.5, 98]
})

print("📊 BEFORE MERGE:")
print("\n5-min DataFrame columns:")
print(df_5min.columns.tolist())
print("\nDaily DataFrame columns:")
print(df_daily.columns.tolist())


📊 BEFORE MERGE:

5-min DataFrame columns:
['datetime', 'Close', 'MA20', 'date']

Daily DataFrame columns:
['datetime', 'ma50', 'ma100', 'ma200']


In [12]:
# ### ❌ WRONG WAY: Merging without renaming causes column conflict


In [13]:
df_wrong = df_5min.copy()
df_daily_wrong = df_daily.copy()

# This merge will create datetime_x and datetime_y
df_merged_wrong = df_wrong.merge(df_daily_wrong, left_on='date', right_on='datetime', how='left')

print("\n❌ WRONG MERGE RESULT:")
print("\nMerged columns:")
print(df_merged_wrong.columns.tolist())
print("\n⚠️ Notice: 'datetime' became 'datetime_x' and 'datetime_y'!")



❌ WRONG MERGE RESULT:

Merged columns:
['datetime_x', 'Close', 'MA20', 'date', 'datetime_y', 'ma50', 'ma100', 'ma200']

⚠️ Notice: 'datetime' became 'datetime_x' and 'datetime_y'!


In [14]:
# ### ✅ CORRECT WAY: Rename before merge to avoid conflict


In [15]:
df_correct = df_5min.copy()
df_daily_correct = df_daily.copy()

# FIX: Rename df_daily['datetime'] to 'date' before merge
df_daily_correct = df_daily_correct.rename(columns={'datetime': 'date'})

# Now merge on 'date' (common column)
df_merged_correct = df_correct.merge(df_daily_correct, on='date', how='left')

print("\n✅ CORRECT MERGE RESULT:")
print("\nMerged columns:")
print(df_merged_correct.columns.tolist())
print("\n✓ 'datetime' column from 5-min data is preserved!")



✅ CORRECT MERGE RESULT:

Merged columns:
['datetime', 'Close', 'MA20', 'date', 'ma50', 'ma100', 'ma200']

✓ 'datetime' column from 5-min data is preserved!


In [16]:
# ## 2. Visualize the Difference


In [17]:
print("\n" + "="*80)
print("COMPARISON: Wrong vs Correct Merge")
print("="*80)

print("\n❌ WRONG MERGE - First 3 rows:")
print(df_merged_wrong.head(3))

print("\n✅ CORRECT MERGE - First 3 rows:")
print(df_merged_correct.head(3))



COMPARISON: Wrong vs Correct Merge

❌ WRONG MERGE - First 3 rows:
           datetime_x  Close   MA20        date  datetime_y  ma50  ma100  \
0 2024-01-01 09:15:00    100   99.0  2024-01-01  2024-01-01  98.0   97.0   
1 2024-01-01 09:20:00    101   99.5  2024-01-02  2024-01-02  98.5   97.5   
2 2024-01-01 09:25:00    102  100.0  2024-01-03  2024-01-03  99.0   98.0   

   ma200  
0   96.0  
1   96.5  
2   97.0  

✅ CORRECT MERGE - First 3 rows:
             datetime  Close   MA20        date  ma50  ma100  ma200
0 2024-01-01 09:15:00    100   99.0  2024-01-01  98.0   97.0   96.0
1 2024-01-01 09:20:00    101   99.5  2024-01-02  98.5   97.5   96.5
2 2024-01-01 09:25:00    102  100.0  2024-01-03  99.0   98.0   97.0


In [18]:
# ## 3. Test Time Extraction
#
# This is what line 154 tries to do: `time = curr_row['datetime'].time()`


In [19]:
print("\n" + "="*80)
print("TIME EXTRACTION TEST")
print("="*80)

# Try to extract time from WRONG merge
try:
    test_row = df_merged_wrong.iloc[0]
    time_value = test_row['datetime'].time()
    print("\n❌ WRONG: Successfully extracted time (shouldn't work)")
    print(f"   Time: {time_value}")
except KeyError as e:
    print(f"\n❌ WRONG: KeyError - {e}")
    print("   ⚠️ 'datetime' column doesn't exist!")
    print(f"   Available columns: {df_merged_wrong.columns.tolist()}")

# Try to extract time from CORRECT merge
try:
    test_row = df_merged_correct.iloc[0]
    time_value = test_row['datetime'].time()
    print(f"\n✅ CORRECT: Successfully extracted time!")
    print(f"   Time: {time_value}")
except KeyError as e:
    print(f"\n✅ CORRECT: KeyError - {e}")



TIME EXTRACTION TEST

❌ WRONG: KeyError - 'datetime'
   ⚠️ 'datetime' column doesn't exist!
   Available columns: ['datetime_x', 'Close', 'MA20', 'date', 'datetime_y', 'ma50', 'ma100', 'ma200']

✅ CORRECT: Successfully extracted time!
   Time: 09:15:00


In [20]:
# ## 4. The Fix Applied to step2_multistock_validation.py
#
# **Line 144-145 (BEFORE):**
# ```python
# df = df.merge(df_daily, left_on='date', right_on='datetime', how='left')
# # Creates datetime_x and datetime_y
# ```
#
# **Line 144-145 (AFTER - FIXED):**
# ```python
# df_daily = df_daily.rename(columns={'datetime': 'date'})  # Rename first
# df = df.merge(df_daily, on='date', how='left')  # Merge on common column
# # Preserves original 'datetime' column
# ```


In [21]:
# ## 5. Summary
#
# | Issue | Wrong Approach | Correct Approach |
# |-------|---------------|------------------|
# | **Merge keys** | `left_on='date', right_on='datetime'` | Rename first, then `on='date'` |
# | **Result** | Creates `datetime_x`, `datetime_y` | Preserves original `datetime` |
# | **Line 154** | ❌ `KeyError: 'datetime'` | ✅ Works perfectly |
#
# **Key Lesson:**
# - Pandas adds `_x` and `_y` suffixes when merging DataFrames with overlapping column names
# - Solution: Rename conflicting columns BEFORE merging to avoid suffix creation
# - This preserves the original column names you need later in the code

